**이력서 텍스트를 받아 정보를 추출하고, 두 가지 분석을 병렬로 실행한 뒤 점수에 따라 면접 안내 또는 불합격 메시지를 만드는 그래프를 만들어라.**
```
RESUMES = [
    "김하늘 / 백엔드 개발 5년. Python(FastAPI), PostgreSQL SQL 튜닝, Docker와 Kubernetes로 배포 자동화. "
    "결제 API 응답 속도 40% 개선.",
    "이도윤 / 프론트엔드 개발 1년. React, TypeScript 사용. 사내 디자인 시스템 컴포넌트 20종 개발.",
]
```
* State(ScreenState): resume, profile, skill_score, strengths, total, decision, message

* 노드

    * extract: with_structured_output(Profile)로 이름 · 경력 연수 · 보유 기술(소문자)을 추출

    * check_skills: 필수 기술 {"python", "sql", "docker"} 일치율 70점 + 경력(3년 이상 30점, 미만은 비례) 점수를 코드로 계산

    * summarize_strengths: LLM으로 강점 한 문장 요약

    * decide: 70점 이상이면 interview, 미만이면 reject

    * invite: 강점을 언급한 3문장 면접 안내 메일(LLM)

    * reject: 부족한 필수 기술을 알려 주는 안내 문구(코드)

* 엣지

    * extract에서 check_skills, summarize_strengths로 엣지를 두 개 내보내 병렬 실행

    * 두 노드가 모두 끝난 뒤 decide 실행

    * decide에서 조건부 엣지로 invite / reject 분기

* draw_mermaid()로 구조를 출력하고, 이력서 2건(합격 · 불합격)을 stream_mode="updates"로 실행해 노드별 변경분을 출력한다.



In [7]:
from typing import Literal, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna", temperature=0, reasoning={"effort": "none"})

# 채점 기준값
NEED = frozenset({"python", "sql", "docker"})
SKILL_MAX, CAREER_MAX, FULL_CAREER_YEARS, PASS_LINE = 70, 30, 3, 70

In [8]:
class Profile(BaseModel):
    name: str = Field(description="지원자 이름")
    career_years: int = Field(description="경력 연수, 정수")
    skills: list[str] = Field(description="보유 기술, 소문자 영문 표기")


class ScreenState(TypedDict, total=False):
    resume: str
    profile: Profile
    skill_score: int
    strengths: str
    total: int
    decision: str
    message: str

In [9]:
def extract(state: ScreenState) -> dict:
    parser = llm.with_structured_output(Profile)
    prof = parser.invoke(
        "이력서에서 이름과 경력 연수, 기술 스택을 뽑아 주세요. 기술은 모두 소문자로 쓰세요.\n"
        f"---\n{state['resume']}"
    )
    prof.skills = [s.strip().lower() for s in prof.skills]
    return {"profile": prof}


def check_skills(state: ScreenState) -> dict:
    prof = state["profile"]
    matched = NEED.intersection(prof.skills)
    skill_pts = SKILL_MAX * len(matched) // len(NEED)
    career_pts = CAREER_MAX * min(prof.career_years, FULL_CAREER_YEARS) // FULL_CAREER_YEARS
    return {"skill_score": skill_pts + career_pts}


def summarize_strengths(state: ScreenState) -> dict:
    reply = llm.invoke(
        "아래 이력서를 읽고 지원자의 가장 큰 강점을 한 문장으로 정리해 주세요.\n"
        f"---\n{state['resume']}"
    )
    return {"strengths": reply.text.strip()}

In [10]:
def decide(state: ScreenState) -> dict:
    score = state["skill_score"]
    return {"total": score, "decision": "interview" if score >= PASS_LINE else "reject"}


def invite(state: ScreenState) -> dict:
    reply = llm.invoke(
        f"지원자 {state['profile'].name}님에게 보내는 면접 안내 메일을 정확히 3문장으로 쓰세요. "
        f"다음 강점을 본문에 언급하세요: {state['strengths']}"
    )
    return {"message": reply.text.strip()}


def reject(state: ScreenState) -> dict:
    prof = state["profile"]
    lacking = sorted(NEED - set(prof.skills))
    reason = ", ".join(lacking) if lacking else "경력"
    return {
        "message": f"{prof.name}님, 이번 채용에는 {reason} 부문이 부족하여 "
                   f"면접 대상에서 제외되었습니다. 지원해 주셔서 감사합니다."
    }

In [11]:
def build_screening_graph():
    g = StateGraph(ScreenState)
    for name, fn in [
        ("extract", extract),
        ("check_skills", check_skills),
        ("summarize_strengths", summarize_strengths),
        ("decide", decide),
        ("invite", invite),
        ("reject", reject),
    ]:
        g.add_node(name, fn)

    g.add_edge(START, "extract")
    # 병렬 분기 후 합류
    for branch in ("check_skills", "summarize_strengths"):
        g.add_edge("extract", branch)
    g.add_edge(["check_skills", "summarize_strengths"], "decide")

    g.add_conditional_edges(
        "decide",
        lambda s: s["decision"],
        {"interview": "invite", "reject": "reject"},
    )
    g.add_edge("invite", END)
    g.add_edge("reject", END)
    return g.compile()


screen_graph = build_screening_graph()
print(screen_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	extract(extract)
	check_skills(check_skills)
	summarize_strengths(summarize_strengths)
	decide(decide)
	invite(invite)
	reject(reject)
	__end__([<p>__end__</p>]):::last
	__start__ --> extract;
	check_skills --> decide;
	decide -. &nbsp;interview&nbsp; .-> invite;
	decide -.-> reject;
	extract --> check_skills;
	extract --> summarize_strengths;
	summarize_strengths --> decide;
	invite --> __end__;
	reject --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [12]:
RESUMES = [
    "김하늘 / 백엔드 개발 5년. Python(FastAPI), PostgreSQL SQL 튜닝, Docker와 Kubernetes로 배포 자동화. "
    "결제 API 응답 속도 40% 개선.",
    "이도윤 / 프론트엔드 개발 1년. React, TypeScript 사용. 사내 디자인 시스템 컴포넌트 20종 개발.",
]

for idx, text in enumerate(RESUMES, 1):
    print(f"\n##### 이력서 {idx} #####")
    for chunk in screen_graph.stream({"resume": text}, stream_mode="updates"):
        for node_name, delta in chunk.items():
            print(f"  -> {node_name}: {delta}")


##### 이력서 1 #####
  -> extract: {'profile': Profile(name='김하늘', career_years=5, skills=['python', 'fastapi', 'postgresql', 'sql', 'docker', 'kubernetes'])}
  -> check_skills: {'skill_score': 100}
  -> summarize_strengths: {'strengths': 'Python 기반 백엔드 개발과 SQL 튜닝, 배포 자동화 역량을 바탕으로 결제 API 성능을 40% 개선한 실무 경험이 가장 큰 강점입니다.'}
  -> decide: {'total': 100, 'decision': 'interview'}
  -> invite: {'message': '김하늘님, 안녕하세요. 면접에 초대드리며, 일정과 접속 방법은 아래 안내를 확인해 주세요. Python 기반 백엔드 개발과 SQL 튜닝, 배포 자동화 역량을 바탕으로 결제 API 성능을 40% 개선하신 실무 경험이 가장 큰 강점이라고 생각합니다.'}

##### 이력서 2 #####
  -> extract: {'profile': Profile(name='이도윤', career_years=1, skills=['react', 'typescript'])}
  -> check_skills: {'skill_score': 10}
  -> summarize_strengths: {'strengths': 'React와 TypeScript를 활용해 사내 디자인 시스템 컴포넌트 20종을 개발한 프론트엔드 구현 역량이 가장 큰 강점입니다.'}
  -> decide: {'total': 10, 'decision': 'reject'}
  -> reject: {'message': '이도윤님, 이번 채용에는 docker, python, sql 부문이 부족하여 면접 대상에서 제외되었습니다. 지원해 주셔서 감사합니다.'}


---

**여러 도시의 1일 일정을 병렬로 생성하고, 총비용이 예산을 넘으면 중요도가 낮은 활동을 하나씩 빼는 조정 루프를 가진 그래프를 만들어라.**

* State(TripState): cities, budget, activities(Annotated[list[dict], operator.add]), total, trimmed, rounds

* 노드

    * fan_out: 도시 수만큼 Send("plan_city", {"city": 도시})를 반환 (START에서 조건부 엣지로 연결)

    * plan_city: with_structured_output(CityPlan)으로 활동 4개(이름 · 비용 · 중요도 1~5) 생성

    * budget_check: 총비용을 계산해 Command(update={"total": ...}, goto=...)로 예산 이내면 END, 초과면 trim으로 이동

    * trim: 중요도가 가장 낮은(동률이면 비싼) 활동 1개를 빼고, 뺀 활동을 trimmed에 기록하고, rounds를 1 늘린다

* 엣지: plan_city → budget_check, trim → budget_check (조정 루프)

* 실행 config에 recursion_limit: 30을 지정한다.

* 교토 · 오사카 · 나라, 예산 15만 원으로 실행해 최종 합계, 조정 횟수, 남은 활동, 제외한 활동을 출력한다.

In [13]:
import operator
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Overwrite, Send
from pydantic import BaseModel, Field

load_dotenv()
llm = ChatOpenAI(model="gpt-6-luna", temperature=0, reasoning={"effort": "none"})

In [14]:
class Spot(BaseModel):
    name: str = Field(description="활동 이름")
    cost: int = Field(description="예상 비용, 원 단위 정수")
    importance: int = Field(description="중요도, 1에서 5 사이 정수")


class CityPlan(BaseModel):
    spots: list[Spot] = Field(description="하루 일정에 넣을 활동 4개")


class TripState(TypedDict, total=False):
    cities: list[str]
    budget: int
    activities: Annotated[list[dict], operator.add]
    total: int
    trimmed: list[str]
    rounds: int

In [15]:
def fan_out(state: TripState) -> list[Send]:
    return [Send("plan_city", {"city": c}) for c in state["cities"]]


def plan_city(payload: dict) -> dict:
    city = payload["city"]
    plan = llm.with_structured_output(CityPlan).invoke(
        f"{city}에서 하루 동안 즐길 활동 4개를 추천하세요. 각 활동의 비용(원)과 "
        f"중요도(1~5, 필수 명소일수록 높음)를 함께 주세요."
    )
    return {
        "activities": [
            {"city": city, "name": s.name, "cost": s.cost, "priority": s.importance}
            for s in plan.spots
        ]
    }

In [16]:
def budget_check(state: TripState) -> Command[Literal["trim", "__end__"]]:
    spent = sum(item["cost"] for item in state["activities"])
    over = spent > state["budget"] and len(state["activities"]) > 0
    return Command(update={"total": spent}, goto="trim" if over else END)


def trim(state: TripState) -> dict:
    # 중요도 오름차순, 동률이면 비용 내림차순으로 가장 앞의 활동 선택
    victim = min(state["activities"], key=lambda a: (a["priority"], -a["cost"]))
    kept = [a for a in state["activities"] if a is not victim]
    note = f"{victim['city']}/{victim['name']}({victim['cost']}원, 중요도 {victim['priority']})"
    return {
        "activities": Overwrite(kept),
        "trimmed": [*state.get("trimmed", []), note],
        "rounds": state.get("rounds", 0) + 1,
    }

In [17]:
trip_builder = StateGraph(TripState)
trip_builder.add_node("plan_city", plan_city)
trip_builder.add_node("budget_check", budget_check)
trip_builder.add_node("trim", trim)
trip_builder.add_conditional_edges(START, fan_out, ["plan_city"])
trip_builder.add_edge("plan_city", "budget_check")
trip_builder.add_edge("trim", "budget_check")
trip_graph = trip_builder.compile()

In [18]:
BUDGET = 150_000
final = trip_graph.invoke(
    {"cities": ["교토", "오사카", "나라"], "budget": BUDGET, "trimmed": [], "rounds": 0},
    config={"recursion_limit": 30},
)

print(f"예산: {BUDGET:,}원")
print(f"최종 합계: {final['total']:,}원")
print(f"조정 횟수: {final['rounds']}회")
print("남은 활동:")
for item in sorted(final["activities"], key=lambda a: (a["city"], -a["priority"])):
    print(f"  - [{item['city']}] {item['name']} | {item['cost']:,}원 | 중요도 {item['priority']}")
print("제외한 활동:")
for note in final["trimmed"]:
    print(f"  - {note}")

예산: 150,000원
최종 합계: 88,000원
조정 횟수: 0회
남은 활동:
  - [교토] 후시미 이나리 신사 산책 | 0원 | 중요도 5
  - [교토] 기요미즈데라와 산넨자카·니넨자카 둘러보기 | 5,000원 | 중요도 5
  - [교토] 니시키 시장에서 교토 음식 맛보기 | 15,000원 | 중요도 4
  - [교토] 아라시야마 대나무숲과 도게쓰교 산책 | 0원 | 중요도 4
  - [나라] 경복궁 관람 | 3,000원 | 중요도 5
  - [나라] 북촌 한옥마을 산책 | 0원 | 중요도 4
  - [나라] 인사동 전통문화 거리 구경 | 0원 | 중요도 4
  - [나라] 남산서울타워 전망대 관람 | 21,000원 | 중요도 4
  - [오사카] 오사카성 천수각 관람 | 6,000원 | 중요도 5
  - [오사카] 도톤보리 산책 및 글리코상 사진 촬영 | 0원 | 중요도 5
  - [오사카] 구로몬 시장에서 현지 음식 맛보기 | 20,000원 | 중요도 4
  - [오사카] 우메다 스카이 빌딩 공중정원 전망대 관람 | 18,000원 | 중요도 4
제외한 활동:
